# Predicting Movie Review Sentiment with BERT (PyTorch + Hugging Face Transformers)

Adapted from Google's [Predicting Movie Reviews with BERT on TF Hub](https://colab.research.google.com/github/google-research/bert/blob/master/predicting_movie_reviews_with_bert_on_tf_hub.ipynb) notebook, rewritten for **PyTorch** and the Hugging Face **`transformers`** library (the original used TensorFlow 1.x APIs that no longer exist).

**BERT** (Bidirectional Encoder Representations from Transformers) is a Transformer encoder pre-trained on a large text corpus with two self-supervised tasks: masked-language modeling and next-sentence prediction. Because it has already learned general-purpose language representations, we can **fine-tune** it on a small labeled dataset and get strong results quickly.

In this notebook we fine-tune `bert-base-uncased` to classify IMDB movie reviews as **positive** or **negative**.

> In Colab, choose **Runtime → Change runtime type → GPU** before running. On a T4 GPU the full run (5,000 training reviews, 3 epochs) takes about 5–10 minutes.

In [ ]:
# Colab already ships with torch and transformers; this makes sure `datasets` is present too.
%pip install -q transformers datasets scikit-learn

In [ ]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():   # Apple-silicon Macs
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

# Data

We load the [IMDB Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/) (25,000 train / 25,000 test reviews) from the Hugging Face Hub. Each example has a `text` field and a `label` (0 = negative, 1 = positive).

To keep training fast, we take a random sample of 5,000 train and 5,000 test examples. On a CPU-only machine we use a smaller sample and 2 epochs so the notebook still finishes in reasonable time (expect lower accuracy).

In [ ]:
imdb = load_dataset("stanfordnlp/imdb")
print(imdb)

if device.type in ("cuda", "mps"):
    N_TRAIN, N_TEST, EPOCHS = 5000, 5000, 3
else:  # CPU only: much slower, so train on less data
    N_TRAIN, N_TEST, EPOCHS = 2000, 1000, 2

train_ds = imdb["train"].shuffle(seed=SEED).select(range(N_TRAIN))
test_ds = imdb["test"].shuffle(seed=SEED).select(range(N_TEST))
print(f"train: {len(train_ds)}  test: {len(test_ds)}  epochs: {EPOCHS}")
print("label balance (train):", np.bincount(train_ds["label"]))

In [ ]:
example = train_ds[0]
print("label:", example["label"])
print(example["text"][:500])

# Data Preprocessing

We need to turn raw text into the input format BERT was pre-trained on:

1. Lowercase the text (we use the *uncased* BERT model).
2. Tokenize it into **WordPiece** tokens, e.g. `"sally says hi" -> ["sally", "says", "hi"]`; rare words are split into sub-word pieces marked with `##`.
3. Add the special tokens: `[CLS]` at the start (its final hidden state is used for classification) and `[SEP]` at the end.
4. Map tokens to integer ids from BERT's vocabulary.
5. Pad or truncate every review to a fixed `MAX_SEQ_LENGTH`, and build an **attention mask** that tells the model which positions are real tokens (1) and which are padding (0).

The tokenizer that matches the pre-trained checkpoint does all of this for us.

In [ ]:
BERT_MODEL = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

print(tokenizer.tokenize("This here's an example of using the BERT tokenizer"))

In [ ]:
enc = tokenizer("That movie was absolutely awful", padding="max_length", max_length=16, truncation=True)
print("input_ids:     ", enc["input_ids"])
print("attention_mask:", enc["attention_mask"])
print("tokens:        ", tokenizer.convert_ids_to_tokens(enc["input_ids"]))

In [ ]:
MAX_SEQ_LENGTH = 128

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_SEQ_LENGTH)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
test_tok = test_ds.map(tokenize, batched=True, remove_columns=["text"])

cols = ["input_ids", "token_type_ids", "attention_mask", "label"]
train_tok.set_format("torch", columns=cols)
test_tok.set_format("torch", columns=cols)

BATCH_SIZE = 32
train_loader = DataLoader(train_tok, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_tok, batch_size=BATCH_SIZE)

batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in batch.items()})

# Creating a model

`AutoModelForSequenceClassification` loads the pre-trained BERT encoder and puts a new, randomly initialized classification head on top:

`[CLS] hidden state (768-d) -> dropout -> linear layer -> 2 logits (negative / positive)`

During fine-tuning, **all** weights (the pre-trained encoder and the new head) are updated with a cross-entropy loss. The warning about newly initialized `classifier` weights is expected, because that head is exactly what we are about to train.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=2).to(device)
print(model.classifier)
print(f"trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

We use the same hyperparameters as the original BERT fine-tuning recipe:

- **AdamW** optimizer with learning rate `2e-5` (fine-tuning uses a small learning rate so we don't destroy the pre-trained weights)
- a **linear warm-up** over the first 10% of steps, then a linear decay to 0

In [ ]:
LEARNING_RATE = 2e-5
WARMUP_PROPORTION = 0.1

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
num_train_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_PROPORTION * num_train_steps),
    num_training_steps=num_train_steps,
)
print("total training steps:", num_train_steps)

# Training

A standard PyTorch training loop. Passing `labels` to the model makes it return the cross-entropy loss along with the logits.

In [ ]:
from time import perf_counter

model.train()
start = perf_counter()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(
            input_ids=batch["input_ids"],
            token_type_ids=batch["token_type_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["label"],
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()
        if step % 20 == 0 or step == len(train_loader):
            print(f"epoch {epoch + 1}  step {step}/{len(train_loader)}  loss {running_loss / step:.4f}")
print(f"Training took {perf_counter() - start:.0f} s")

# Evaluation

Now let's use our held-out test reviews to see how well the model did.

In [ ]:
@torch.no_grad()
def predict_proba(loader):
    model.eval()
    probs, labels = [], []
    for batch in loader:
        labels.append(batch.pop("label"))
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu())
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

p_pos, y_true = predict_proba(test_loader)
y_pred = (p_pos >= 0.5).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
print(f"accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"precision: {precision:.4f}")
print(f"recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"AUC      : {roc_auc_score(y_true, p_pos):.4f}")
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")

# Predicting on new sentences

Finally, a helper that tokenizes raw strings and returns the predicted label and probability.

In [ ]:
LABELS = ["Negative", "Positive"]

@torch.no_grad()
def get_prediction(sentences):
    model.eval()
    enc = tokenizer(sentences, padding=True, truncation=True, max_length=MAX_SEQ_LENGTH, return_tensors="pt").to(device)
    probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
    return [(s, LABELS[p.argmax()], round(float(p.max()), 4)) for s, p in zip(sentences, probs)]

pred_sentences = [
    "That movie was absolutely awful",
    "The acting was a bit lacking",
    "The film was creative and surprising",
    "Absolutely fantastic!",
]
for sentence, label, prob in get_prediction(pred_sentences):
    print(f"{label:8s} ({prob:.3f})  {sentence}")

Voila! We have a sentiment classifier!

**Things to try:**
- Increase `N_TRAIN`, `EPOCHS`, or `MAX_SEQ_LENGTH` and see how accuracy changes (and how long training takes).
- Swap `BERT_MODEL` for `distilbert-base-uncased` (smaller and faster) or `roberta-base`. Note that DistilBERT and RoBERTa have no `token_type_ids`, so drop that column from `cols` and from the training-loop call.
- Freeze the encoder (`for p in model.bert.parameters(): p.requires_grad = False`) and train only the classification head. How much worse is it than full fine-tuning?